# Nhiệm vụ 1.3: Phân tích Mức độ Thiếu Dữ liệu (Missingness Profile) & Bóc tách Ma trận 2 chiều (Source x Category)

**Thành viên thực hiện**: TV3 (Exploratory Data Analysis)

**Tập dữ liệu đầu vào**: `data/processed/products_processed.jsonl` (10.316 bản ghi, 30 thuộc tính chuẩn hóa)

**Mục tiêu**:
1. Khảo sát tỷ lệ missing (%) của toàn bộ 30 cột và phân loại mức độ nghiêm trọng.
2. Phân rã tỷ lệ missing theo 2 chiều: **Nguồn cào (Source)** x **Ngành hàng (Category)**.
3. Chứng minh hiện tượng **Thiếu mang tính Cấu trúc (Structural Missingness / MNAR)**: tại sao một số cột chỉ có ở Laptop, một số cột chỉ có ở Smartphone, và một số cột chỉ có ở một vài sàn cụ thể.
4. Cung cấp căn cứ thực nghiệm để TV5 xây dựng chiến lược Imputation phù hợp.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent / "scripts"))
sys.path.append("src/eda/scripts")

from eda_stats import (
    load_processed_products,
    compute_overall_missingness,
    compute_missingness_by_category,
    compute_missingness_by_source,
    compute_missingness_matrix_source_category
)
from eda_viz import (
    set_plotting_theme,
    plot_overall_missingness_ranking,
    plot_missingness_heatmap_by_source_category,
    plot_hardware_missingness_comparison
)

set_plotting_theme()
print("Môi trường phân tích sẵn sàng!")

## 1. Tải dữ liệu và Xếp hạng Tỷ lệ Thiếu của Toàn bộ 30 Thuộc tính

In [ ]:
df = load_processed_products("data/processed/products_processed.jsonl")
overall_miss_df = compute_overall_missingness(df)
display(overall_miss_df)

In [ ]:
fig1_path = "src/eda/outputs/figures/fig_1_3_overall_missingness_ranking.png"
plot_overall_missingness_ranking(overall_miss_df, fig1_path)
print(f"Đã xuất biểu đồ: {fig1_path}")

## 2. So sánh Tỷ lệ Thiếu giữa Laptop và Smartphone — Khám phá Đặc thù Ngành hàng

In [ ]:
cat_miss_df = compute_missingness_by_category(df)
display(cat_miss_df)

In [ ]:
fig3_path = "src/eda/outputs/figures/fig_1_3_hardware_missingness_laptop_vs_phone.png"
plot_hardware_missingness_comparison(df, fig3_path)
print(f"Đã xuất biểu đồ: {fig3_path}")

## 3. Ma trận Biểu đồ Nhiệt (Heatmap) Phân rã 2 chiều: Source x Category

In [ ]:
source_miss_df = compute_missingness_by_source(df)
display(source_miss_df)

matrix_2d_df = compute_missingness_matrix_source_category(df)
display(matrix_2d_df)

In [ ]:
fig2_path = "src/eda/outputs/figures/fig_1_3_missingness_heatmap_by_source_category.png"
plot_missingness_heatmap_by_source_category(df, fig2_path)
print(f"Đã xuất biểu đồ: {fig2_path}")

## 4. Kết luận cốt lõi & Khuyến nghị Tiền xử lý cho TV5

1. **Nhóm đặc trưng hoàn chỉnh (Missing 0.0% – 0.3%)**:
   * `listing_record_id`, `source_parent_id`, `product_name`, `source`, `category`, `seller_raw`, `availability_status`, `is_price_missing`, `has_rating` (100% đầy đủ).
   * `price_vnd` (chỉ thiếu 0,16% — 16 bản ghi lỗi).
   * `brand_norm` (chỉ thiếu 0,27% — 28 bản ghi).
   * *Khuyến nghị*: Đây là bộ khung chuẩn hóa cực kỳ vững chắc.

2. **Hiện tượng Thiếu mang tính Cấu trúc theo Ngành hàng (Category-Specific Structural Missingness)**:
   * `battery_wh` (Laptop thiếu 59,2%, Smartphone thiếu 99,9%).
   * `battery_mah` (Laptop thiếu 99,4%, Smartphone thiếu 24,1%).
   * `gpu_norm` (Laptop thiếu 15,3%, Smartphone thiếu 70,0%).
   * `cpu_norm` (Laptop thiếu 21,7%, Smartphone thiếu 82,5%).
   * `camera_mp` (Laptop thiếu 97,1%, Smartphone thiếu 87,6%).
   * *Khuyến nghị cho TV5*: **Bắt buộc phải tách thành 2 mô hình huấn luyện độc lập**: 1 mô hình cho Laptop và 1 mô hình cho Smartphone. Không được gộp chung rồi cố điền Pin Wh cho Smartphone hay điền Camera MP cho Laptop.

3. **Hiện tượng Thiếu mang tính Cấu trúc theo Sàn cào (Source-Specific Missingness)**:
   * `total_sold_normalized`: Chỉ tồn tại ở **FPT Shop** (thiếu 6,5%) và **TGDĐ** (thiếu 5,0%). Thiếu 100% tại CellphoneS, Hoàng Hà, Viettel Store do các website này không công khai số lượng đã bán.
   * `charger_watt`: Thiếu 100% tại CellphoneS và FPT Shop, nhưng có ở TGDĐ và Viettel Store.
   * `rating_norm`: Thiếu 100% tại Hoàng Hà Mobile do sàn không có hệ thống đánh giá sao công khai, nhưng ở TGDĐ chỉ thiếu 4,8%.
   * *Khuyến nghị cho TV5*: Tuyệt đối không dùng `total_sold_normalized` làm đặc trưng dự đoán giá chung nếu mô hình áp dụng đa sàn, vì sẽ gây ra hiện tượng rò rỉ thiên lệch sàn (Source Bias Leakage).